# Activity 2: Async, SDKs, and the AI Era

**Module:** Week 2 Day 3
**Estimated time:** 70 to 90 minutes
**Type:** Guided notebook
**Libraries:** `httpx`, `asyncio`, `boto3`, `google-genai`, `python-dotenv`
**AI-free zone note:** calling an AI service through its API is allowed and is
the point of this activity. The rule you follow is simpler: do not use an AI
assistant to write your code for you. Here you learn how API calls work, using
an AI service as one example among several.

## What you will learn

- The difference between calling several APIs one at a time and calling them
  concurrently, and why it matters for data ingestion
- How to use `httpx.AsyncClient` with `asyncio.gather`
- The difference between an API and an SDK, and why cloud providers ship SDKs
- That modern AI services (and MCP, the protocol behind AI agent tools) are
  just API calls underneath

## Setup

Activity 0 created the shared environment at the repo root and you selected its
`.venv` as this notebook's kernel, so these are already installed. If one is
missing, add it from the repo root:

```bash
uv add httpx boto3 google-genai python-dotenv
```

You need a `GOOGLE_API_KEY` for the Gemini section (the AWS section below
reads a public S3 object anonymously, so it needs no AWS credentials at all).
Put the Gemini key in your `.env` file:

```text
GOOGLE_API_KEY=your-key-here
```

Never paste a key directly into a code cell.

## 1. Several calls, one at a time (sync)

In Activity 1 you fetched 8 cities from Open-Meteo in a plain loop, with a
`time.sleep(0.3)` between each one so you never hammered the API. Here you
time that exact pattern, then compare it to running the same 8 calls
concurrently.

In [ ]:
WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
CITIES = {
    "Hartford,US":  (41.7658, -72.6734),
    "New York,US":  (40.7128, -74.0060),
    "Boston,US":    (42.3601, -71.0589),
    "Chicago,US":   (41.8781, -87.6298),
    "Denver,US":    (39.7392, -104.9903),
    "Seattle,US":   (47.6062, -122.3321),
    "Miami,US":     (25.7617, -80.1918),
    "Phoenix,US":   (33.4484, -112.0740),
}

In [ ]:
import time
import httpx

start = time.perf_counter()
with httpx.Client(timeout=10) as client:
    for city, (lat, lon) in CITIES.items():
        resp = client.get(WEATHER_URL, params={
            "latitude": lat, "longitude": lon,
            "current": "temperature_2m", "temperature_unit": "fahrenheit",
        })
        resp.raise_for_status()
        temp = resp.json()["current"]["temperature_2m"]
        print(f"  {city}: {temp}F")
sync_seconds = time.perf_counter() - start
print(f"\nsync total: {sync_seconds:.2f}s")

**Expected output shape** (real temperatures will vary):

```text
  Hartford,US: 71.6F
  New York,US: 74.1F
  ...

sync total: 1.9s
```

Eight calls, each waiting on the network before the next starts. The total
time is roughly the sum of all eight individual wait times, which is exactly
why the Activity 1 version needed patience (and the polite `time.sleep(0.3)`
made it a little slower still, on purpose, to avoid hammering a free API).

## 2. The same calls, concurrently (async)

`async`/`await` lets your program start a call, and while it waits for the
network, go start the next one instead of sitting idle. Jupyter lets you use
`await` directly at the top level of a cell. Start with the smallest possible
version: a single async call, run directly, no function yet.

In [ ]:
async with httpx.AsyncClient(timeout=10) as client:
    resp = await client.get(WEATHER_URL, params={
        "latitude": 41.7658, "longitude": -72.6734,
        "current": "temperature_2m", "temperature_unit": "fahrenheit",
    })
    resp.raise_for_status()
    print("Hartford,US:", resp.json()["current"]["temperature_2m"], "F")

That is one async call. You need to do this 8 times, one per city, so wrap
the single-call logic in a coroutine function before you try to run several
of them together.

In [ ]:
async def fetch(client, city, lat, lon):
    resp = await client.get(WEATHER_URL, params={
        "latitude": lat, "longitude": lon,
        "current": "temperature_2m", "temperature_unit": "fahrenheit",
    })
    resp.raise_for_status()
    return city, resp.json()["current"]["temperature_2m"]

# PAUSE: running this cell does nothing yet. It only defines the function;
# nothing calls it. That surprises people the first time.

Now run that coroutine 8 times *concurrently* with `asyncio.gather`, and
time it.

In [ ]:
import asyncio

start = time.perf_counter()

async with httpx.AsyncClient(timeout=10) as client:
    coros = [fetch(client, city, lat, lon) for city, (lat, lon) in CITIES.items()]
    results = await asyncio.gather(*coros)

async_seconds = time.perf_counter() - start
for city, temp in results:
    print(f"  {city}: {temp}F")
print(f"\nasync total: {async_seconds:.2f}s")

**Expected output**: the same 8 cities and temperatures as the sync run,
but `async_seconds` should be noticeably smaller than `sync_seconds`, closer to
the time of the single slowest call than to the sum of all eight.

## 3. What just happened, and why it matters

Sync waited for each call in turn: total time is roughly the sum of every
call's wait. Async overlapped the waiting: total time is roughly the slowest
single call, because the other seven were waiting in the background at the
same time. This is the exact same 8-city fetch from Activity 1, just run a
different way, which is the point: async is not a new dataset or a new API,
it is a different way of waiting.

**Async helps** when your program spends its time waiting on the network or
disk, which describes most data ingestion: calling ten API endpoints, reading
twenty files, querying a slow database. **Async does not speed up** CPU-bound
work like heavy number crunching. There is nothing to overlap if the CPU
itself is the bottleneck, not the waiting.

This is the same model FastAPI uses, which you will meet later in the course.
When an async FastAPI endpoint is waiting on a database or another API, the
server is free to handle other requests in the meantime, instead of blocking
everyone behind that one slow call.

**PAUSE**: if you had 100 cities to call instead of 8, and each one took
about half a second, roughly how long would the sync version take? Would the
async version scale the same way?

## 4. API versus SDK

- An **API** is the contract: send an HTTP request to a URL, get a response
  back. You saw this in Activity 1 with raw `requests`/`httpx` calls.
- An **SDK** (software development kit) is a language-specific wrapper around
  an API. It handles the parts that are easy to get wrong by hand:
  authentication, request signing, retries, and pagination.

You can almost always do the same work either way. The SDK is usually shorter
and safer for production, because the provider has already solved the hard
parts for you.

**Why AWS ships boto3**: an authenticated AWS request must be signed with AWS
Signature Version 4, several careful steps of hashing and header construction.
You could implement that by hand over raw HTTP, but nobody does, because
`boto3` does it for you automatically on every authenticated call (the kind
you used in Week 1's Console/CloudShell labs with your assigned IAM role).

Today's example reads one **public, read-only** object from S3:
`s3://tscookbook/AirQualityUCI.csv`, a course dataset. Because the object is
public, there is no signing to demonstrate, so we explicitly tell boto3 *not*
to sign the request (`UNSIGNED`). The teaching point is narrower but still
real: even a "no auth" read still goes through a proper SDK client and typed
methods (`head_object`, `get_object`), not a hand-built raw HTTP request
against S3's XML-based REST API, which is possible but genuinely painful.

Notice we do **not** list the bucket. Listing needs a separate permission
(`s3:ListBucket`) that a public object does not grant, and you do not need it
here: you already know the exact key you want. Ask only for what you need.

In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

# UNSIGNED means "do not sign this request." No AWS keys are read or needed,
# because this specific object is public and read-only.
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

BUCKET = "tscookbook"
KEY = "AirQualityUCI.csv"
print("client ready; nothing fetched yet")

In [ ]:
# head_object asks only for metadata (the "receipt"), not the file itself.
# This is the same read-the-receipt-before-the-body idea from Activity 1.
head = s3.head_object(Bucket=BUCKET, Key=KEY)
print("content type: ", head["ContentType"])
print("size (bytes): ", f"{head['ContentLength']:,}")
print("last modified:", head["LastModified"])

In [ ]:
# Now fetch the object body. .read() returns bytes; decode to text.
obj = s3.get_object(Bucket=BUCKET, Key=KEY)
raw_bytes = obj["Body"].read()
text = raw_bytes.decode("utf-8")

lines = text.splitlines()
print("total lines:    ", len(lines))
print("header row:     ", lines[0][:70], "...")
print("first data row: ", lines[1][:70], "...")

**Expected output shape** (this is the UCI Air Quality dataset, a real,
semicolon-delimited CSV):

```text
content type:  text/csv
size (bytes):  785,065
last modified: 2021-05-03 02:21:30+00:00

total lines:     9472
header row:      Date;Time;CO(GT);PT08.S1(CO);NMHC(GT);C6H6(GT);PT08.S2(NMHC);NOx(GT) ...
first data row:  10/03/2004;18.00.00;2,6;1360;150;11,9;1046;166;1056;113;1692;1268;13, ...
```

Notice there is no signing code, no manual header construction, and no `.env`
file for this call: `boto3.client("s3", config=...)` built the client, and
`head_object(...)` / `get_object(...)` are single method calls that each map to
one HTTP request underneath (here, explicitly unsigned). `head_object` fetched
only the cheap metadata before you committed to downloading the full object
with `get_object`: the same "check before you pull" discipline you used with
status codes in Activity 1.

This CSV is intentionally messy (semicolons as separators, commas as decimal
points, trailing empty columns). You will meet exactly this kind of real-world
mess when you clean data with pandas and Polars on Day 4.

## 5. An AI service is just an API: Gemini through the SDK

We use the `google-genai` SDK (the current, supported library for 2026; the
older `google-generativeai` package is deprecated and you should never install
it). Like `boto3`, it reads configuration, here your API key, from the
environment.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print("key loaded:", "GOOGLE_API_KEY" in os.environ)

In [ ]:
from google import genai

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="In one sentence, what is a REST API?",
)
print(response.text)

**Expected output shape** (wording will vary): a single sentence defining a
REST API in the model's own words.

## 6. The same AI call as a raw API request

The SDK call above is a convenience. Underneath it is one HTTP POST. Seeing
the raw version makes the point concrete: an AI service is an API like any
other, with a URL, a header carrying your key, and a JSON body.

In [ ]:
import httpx

GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent"

resp = httpx.post(
    GEMINI_URL,
    headers={"x-goog-api-key": os.environ["GOOGLE_API_KEY"]},
    json={"contents": [{"parts": [{"text": "In one sentence, what is a REST API?"}]}]},
    timeout=10,
)
resp.raise_for_status()

In [ ]:
text = resp.json()["candidates"][0]["content"]["parts"][0]["text"]
print(text)

**Expected output**: the same kind of one-sentence answer as the SDK call
above, because it is the same underlying HTTP call. Compare the two: the SDK
version hid the URL, the header name, and the exact JSON shape of the request
body. That is the entire value an SDK adds: convenience and safety, not a
different capability.

## The punchline: APIs in the AI era

People sometimes ask why API skills still matter when an AI tool can write
code for you. The answer is in what you just built: the AI tool itself is
reached through an API. So is AWS. So is Google Cloud. So is every weather,
maps, or payments service you will ever integrate.

Even **MCP** (Model Context Protocol), which lets AI agents call external
tools, is API calls underneath. The pipelines you build for the rest of this
course, and the rest of your career, are made of API calls wrapped in
increasingly convenient layers. Knowing how a request, a response, an SDK,
and an endpoint fit together is the skill that does not expire.

## Checkpoint

State, out loud to a neighbor, one thing the `boto3` or `google-genai` SDK did
for you that the raw `httpx` call had to do by hand. Then confirm your async
run in Section 2 was faster than your sync run in Section 1.